# Hebrew Acronym Disambiguation: Experimental Study

This notebook combines the development protocol with analysis of separately identified saved test runs. Default execution makes no model requests. Sections 1–10 retain the development preview; Section 11 reports collected test evidence. Collection dates, versions, task coverage, and limitations remain explicit rather than implying one uninterrupted run.

## Development protocol and preview

How do trained DictaBERT, Qwen and Gemini resolve a marked Hebrew acronym in context? We compare five arms: DictaBERT selection, and generation and selection by each LLM. All arms use the same occurrences marked `[ACR]...[/ACR]`; the LLMs share prompts and candidate orders.

Follow [local setup and run instructions](../README.md#install-and-check), then set the paths and enable each system below. Run cells in order. `preview` uses invented examples; `run` executes enabled systems; `reload` opens a saved run. Use a new run ID after changing settings.

In [ ]:
from pathlib import Path
from uuid import uuid4
from IPython.display import HTML, display
from hebrew_acronyms import experimental_study as study
from hebrew_acronyms.paper_export import export_paper
from hebrew_acronyms.saved_study_comparison import display_saved_comparison

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
settings = dict(
    mode="preview", run_kind="validation", run_id=str(uuid4()), root=ROOT,
    train_path=ROOT / "data/study_v1/encoder_inputs/train.csv",
    input_path=ROOT / "data/study_v1/encoder_inputs/dev.csv", expected_dev_items=62,
    validation_items=3, device="cpu", checkpoint=None, snapshot_path=None,
    checkpoint_format="manifest", checkpoint_sha256=None, checkpoint_attestation=None,
    enable_encoder=False, saved_encoder=None, saved_encoder_run_id=None,
    enable_qwen=False, qwen_model="qwen2.5:7b", ollama_url="http://localhost:11434",
    qwen_options={}, enable_gemini=False, gemini_model="gemini-3.8-flash",
    gemini_generation_config={"thinkingConfig": {"thinkingLevel": "low"}},
    request_timeout=120, output_root=ROOT.parent / "artifacts/study-runs",
    saved_run=None, saved_run_id=None,
    paper_source_run_id=None,  # Set the exact selected full-dev run ID to enable paper export.
)


## 1. Inputs

The qualified dev set contains 62 items. `validation` selects three and limits each enabled LLM to one request per task; `full_dev` uses all items. The saved cohort identifies the selected rows and the full dev input.

In [ ]:
rows, cohort = study.read_study_inputs(settings)
artifact = study.prepare_study(rows, cohort, settings)
run = artifact["settings"]
request_limit = 1 if run.get("run_kind") == "validation" else None
print("Run:", artifact["run_id"], "Items:", len(rows))

## 2. DictaBERT selection

The default manifest route compares original train/full-dev identities before inference. For the explicitly authorized complete Colab state dictionary, set `checkpoint_format="colab_state_dict"`, the checkpoint and local snapshot paths, its authorized `checkpoint_sha256`, and the supplied `checkpoint_attestation`. The separate adapter requires a full strict load and validates three items before continuing. Its reconstruction evidence records unknown training provenance without inventing an original manifest.


In [ ]:
encoder_predictions = None
if settings["mode"] == "run" and (run["enable_encoder"] or run["saved_encoder"]):
    if run.get("checkpoint_format") == "colab_state_dict":
        encoder_predictions = study.predict_colab_study(
            artifact, checkpoint=run["checkpoint"], snapshot_path=run["snapshot_path"],
            expected_sha256=run["checkpoint_sha256"], device=run["device"],
        )
    else:
        encoder_predictions = study.predict_encoder(
            artifact, rows, checkpoint=run["checkpoint"], snapshot_path=run["snapshot_path"],
            device=run["device"], saved_path=run["saved_encoder"], saved_run_id=run["saved_encoder_run_id"],
        )
checkpoint_info, checkpoint_summary = study.checkpoint_report(artifact)
checkpoint_summary

## 3. Qwen generation

Generate an expansion from the marked sentence and target, without candidates or gold.

In [ ]:
if settings["mode"] == "run" and run["enable_qwen"]:
    qwen = study.connect_qwen(artifact)
    study.collect_responses(artifact, "generate", qwen, system="qwen",
                            limit=request_limit, on_record=study.save_study)

## 4. Qwen selection

Choose from shuffled candidates. A completed response must contain one uppercase, in-range letter after trimming. Incomplete responses retain their text and stopping reason but count as failures.

In [ ]:
if settings["mode"] == "run" and run["enable_qwen"]:
    study.collect_responses(artifact, "select", qwen, system="qwen",
                            limit=request_limit, on_record=study.save_study)

## 5. Gemini generation

Use the same generation prompt as Qwen. Only final text parts are collected; returned model versions are checked across both tasks.

In [ ]:
if settings["mode"] == "run" and run["enable_gemini"]:
    gemini = study.connect_gemini(artifact)
    study.collect_responses(artifact, "generate", gemini, system="gemini",
                            limit=request_limit, on_record=study.save_study)

## 6. Gemini selection

Use Qwen's prompt and candidate order with the same selection parser. Each provider runs independently.

In [ ]:
if settings["mode"] == "run" and run["enable_gemini"]:
    study.collect_responses(artifact, "select", gemini, system="gemini",
                            limit=request_limit, on_record=study.save_study)

## 7. Save outputs

Each attempt saves model metadata, settings, prompts, responses and failures in its run directory. This cell returns the artifact path for later reload.

In [ ]:
artifact_path = None
if settings["mode"] == "run":
    artifact_path = study.save_study(artifact)
print("Saved artifact:", artifact_path)

## 8. Results

These are preliminary **dev** results. Selection uses exact candidate–gold equality after trimming. Micro accuracy divides correct predictions by all requested items; macro averages within-type accuracies. Failures and unrun items remain in the denominator of an attempted system. An entirely unrun system is not measured; missing type metadata makes macro unavailable. Generation requires human judgment.

Expand items to inspect context, candidates, responses and failures. Disagreements compare valid selections. Change `start` to view more items; `results` contains all derived results. Training, architecture and service differences mean this comparison does not isolate the effect of model size.

In [ ]:
prediction_view, results = study.display_tables(artifact, start=0, max_items=10)
display(HTML(prediction_view))
metrics = results["metrics"]
disagreements = results["disagreements"]

## 9. Export the selected paper source

Set `paper_source_run_id` above to the exact full-dev run ID chosen for the paper. This cell then exports the current artifact, including after `reload`, without making new model calls. Preview, validation and invented fixtures cannot overwrite paper outputs. Selection numbers come from the same shared scorer as the display above; failures and unrun items remain in its denominator, partial coverage is labelled, and unjudged generation stays unavailable.

The atomic `paper/generated/current.tex` include selects one immutable bundle of PDF, table, text macros and a manifest with run, input, model and content hashes. Replacing the selected run updates these together; it does not update scientific conclusions. Build instructions verify the bundle before compilation. No export occurs until a source run is selected explicitly.


In [ ]:
paper_manifest = None
if settings["paper_source_run_id"] is not None:
    paper_manifest = export_paper(
        artifact, expected_run_id=artifact["run_id"],
        paper_source_run_id=settings["paper_source_run_id"],
        output_dir=ROOT / "paper/generated",
    )
    print("Paper source:", paper_manifest["run_id"], paper_manifest["execution_status"])


### Reproducibility

Saved artifacts support inspection without the original research files. See the [training appendix](train_dictabert.ipynb) for model fitting. Mock tests verify code behavior; local validation must establish checkpoint and service compatibility.

AI assistance contributed code, documentation and mock checks. Human annotation and scientific validation remain the researchers' responsibility.

## 10. Compare separate saved runs
Set each original `study.json` path and exact run ID below to inspect the five arms together. This cell only reads saved responses, checks that full input identities and LLM prompts agree, and preserves the original arm run IDs. It never starts models or requests. All 62 items are available for review with Ben; generation remains unjudged.

The explicit Colab state-dictionary adapter in `models/dictabert_cross_encoder/colab.py` supports the authorized complete fp16 weights without inventing an original manifest. It records strict key loading, the reconstructed tokenizer/architecture and unresolved training provenance separately. The manifest-bound loader remains available.


In [ ]:
saved_sources = []  # [(Path("/path/to/study.json"), "exact-original-run-id"), ...]
if saved_sources:
    comparison_view, comparison = display_saved_comparison(saved_sources)
    display(HTML(comparison_view))


## 11. Saved test evaluation

This section compares identified, saved **test** runs independently of the development study above. It reads artifacts and makes no model requests. All 395 test items remain in both task denominators, including incomplete responses. The raw runs remain separate.

Original collection uses `a319f9d`; subsequent xAI/Qwen14 collection uses separately identified revisions. Unpack each identified backup into its directory below, or use its original Drive directory. Session and run identities are checked before comparison, followed by input snapshots, protocol, prompts, candidate order, and baseline provenance. The loader reuses the unchanged collection scoring implementation.

Local analysis requires the installed project and Jupyter notebook tools. The default development preview above uses invented examples; its outputs are not test observations.

Saved analysis is enabled automatically when the supplied artifact directory exists. With no saved bundle, the development preview remains independently runnable. Missing required run identities or mismatched artifacts raise an error rather than substituting results.

In [ ]:
SAVED_ARTIFACTS = ROOT / "saved-results"
if not SAVED_ARTIFACTS.is_dir():
    SAVED_ARTIFACTS = ROOT.parent / "artifacts"
RUN_SAVED_TEST_ANALYSIS = globals().get("RUN_SAVED_TEST_ANALYSIS", (SAVED_ARTIFACTS / "colab-runs").is_dir())
if not RUN_SAVED_TEST_ANALYSIS:
    print("Saved test artifacts unavailable: set SAVED_ARTIFACTS to the supplied results bundle.")

if RUN_SAVED_TEST_ANALYSIS:
    from hebrew_acronyms.saved_test_comparison import compare_saved_tests, result_table
    import matplotlib.pyplot as plt
    from html import escape
    import json

    TEST_ARTIFACTS = SAVED_ARTIFACTS / "colab-runs"
    TEST_SESSIONS = [
        (TEST_ARTIFACTS / "evaluation-20261009-a319f9d/extracted",
         "8aa951a4ac96c126335f40ddc869cc7b6b6010cc94d7be1cc30925f09be9aac0"),
        (TEST_ARTIFACTS / "evaluation-gemini-retry-20261009-a319f9d/extracted",
         "837973fb8065fa7c6467d9370fb170ad58df757d3dadd33ae105910ecd245deb"),
        (TEST_ARTIFACTS / "evaluation-xai-qwen14-20261009-64f47b7/extracted",
         "f373731d81617852b35578fb78ab9d222598723f68e4907e9c5127392ebb5b64"),
        (TEST_ARTIFACTS / "evaluation-xai-qwen14-20261009-e9c8ef7/qwen-full-backup",
         "2b185ec5d6baf1975629d82ec4608da353c376aaadebfbe7a483e80f5e6ee2f3"),
    ]
    TEST_RUN_IDS = {
        "qwen": "cd1331b460364d4ebc341194c07dc1fe",
        "qwen14": "f23e084aace743e8a61a1a71dfbb33d4",
        "openai": "358e1fe65fa048c191eda08c078fd4cd",
        "anthropic": "3248737a597b435181227711caea89d0",
    }
    test_comparison = compare_saved_tests(TEST_SESSIONS, TEST_RUN_IDS, diagnostic_sources=[
        (TEST_ARTIFACTS / "evaluation-xai-qwen14-20261009-64f47b7/diagnostic/extracted",
         "e1436f143f73d846412d48f537cb8592afe1332de2879feba7f1572aae931d28",
         "6b3e9430e2b9f9b5f0fb298b87447b48f4eefa8fb7c659a1b7981602297deb77"),
    ])
    print("Validated test inputs:", test_comparison["test_source_sha256"])
    print("Distinct journalled attempts, including pilots:", test_comparison["unique_attempts"])

    from hebrew_acronyms.encoder_test_results import load_encoder_test
    encoder_test = load_encoder_test(
        SAVED_ARTIFACTS / "study-runs/dictabert-test-20261009-f49c3b5",
        ROOT / "data/splits/test_items.csv",
        expected_run_id="dictabert-test-20261009-f49c3b5",
        expected_identity="2d3a20743715252b1ed743458f8efe728743ec8502329cca517f5e746cd3a126",
        expected_predictions_sha256="51050d90e9067c50077f14cea660c3a4ceee761044f514665ad1066cd80187b8",
    )
    combined_test_metrics = test_comparison["metrics"] + [encoder_test["metric"]]


### 11.1 Systems and collection provenance

Model settings are shown as recorded, including output limits and thinking settings. Omitted provider defaults are not inferred. Run identifiers refer to the original collection, not to this analysis. Environment differences and model families prevent interpreting these results as an isolated causal effect of model size.

In [ ]:
if RUN_SAVED_TEST_ANALYSIS:
    display(HTML(result_table(test_comparison["models"], {
        "system": "System", "model": "Requested model", "returned_models": "Returned model",
        "run_id": "Original test run ID", "calls": "API/local calls", "seconds": "Request time (s)",
    })))
    for model in test_comparison["models"]:
        details = {key: model[key] for key in (
            "collection_revision", "identity_sha256", "settings", "usage", "finish_reasons",
            "unverified", "manifest_path", "manifest_sha256")}
        display(HTML("<details><summary>" + escape(model["system"]) +
                     " — settings and evidence</summary><pre style='white-space:pre-wrap;overflow-wrap:anywhere'>" +
                     escape(json.dumps(details, ensure_ascii=False, indent=2)) + "</pre></details>"))


### 11.2 Coverage and automatic scores

`historical_quote_normalized_gold_substring` is the existing string-based generation rule. `strict_candidate_label_v1` parses the selected label and compares its candidate with the stored gold expansion. Neither is a human semantic judgment. Each accuracy uses **all 395 items**; incomplete or absent responses do not disappear from the denominator. “Valid” retains its original task-specific definition and is not technical completion.
The trained DictaBERT arm performs selection only and uses `exact_candidate_match`. Its target marker follows the explicitly selected first-occurrence rule; the LLMs receive the acronym name in their prompts. No generation score is assigned to the encoder.


In [ ]:
if RUN_SAVED_TEST_ANALYSIS:
    display(HTML(result_table(combined_test_metrics, {
        "system": "System", "task": "Task", "n_items": "All items",
        "n_completed": "Technically complete", "n_correct": "Automatically correct",
        "accuracy": "Automatic accuracy", "n_valid": "Valid under stored rule",
        "score": "Scoring rule", "status_counts": "All response statuses",
    })))


In [ ]:
if RUN_SAVED_TEST_ANALYSIS:
    system_order = [name for name in ("qwen", "qwen14", "openai", "anthropic", "xai", "gemini") if name in TEST_RUN_IDS]
    model_labels = {m["system"]: m["model"] for m in test_comparison["models"]}
    model_labels["dictabert"] = "Trained DictaBERT"
    model_colors = {"qwen": "#3569A8", "openai": "#B66A27", "anthropic": "#647844", "gemini": "#946A91", "dictabert": "#53636D", "qwen14": "#61A0C0", "xai": "#A75861"}
    with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 11, "axes.spines.top": False,
                         "axes.spines.right": False, "figure.facecolor": "white"}):
        fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), layout="constrained", sharex=True)
        for ax, task, title in zip(axes, ("generate", "select"), ("Free generation", "Candidate selection")):
            groups = {m["system"]: m for m in combined_test_metrics if m["task"] == task}
            task_order = system_order + (["dictabert"] if task == "select" else [])
            values = [groups[name]["accuracy"] * 100 for name in task_order]
            bars = ax.barh([model_labels[name] for name in task_order], values,
                           color=[model_colors[name] for name in task_order], height=0.58)
            ax.bar_label(bars, labels=[f"{v:.1f}%" for v in values], padding=5)
            ax.invert_yaxis()
            ax.set_xlim(0, 100)
            ax.set_xlabel("Automatic accuracy (%) — all 395 items")
            ax.set_title(title)
            ax.xaxis.grid(True, color="#E5E7EB")
            ax.set_axisbelow(True)
        fig.suptitle("Saved test results · task-specific automatic scoring")
        plt.show()


### 11.3 Baselines and incomplete responses

The four saved candidate-selection baselines use the same test input. Uniform random is an analytical expectation, not a sampled run; oracle assumes the gold answer can be selected. The saved baseline summary uses four decimal places. These are not generation systems.

All incomplete model responses are retained below with their original run and item identifiers. Their full raw content remains in the journals; the table shows a bounded preview. No selective completion or replacement has been performed.

In [ ]:
if RUN_SAVED_TEST_ANALYSIS:
    baseline_values = test_comparison["baselines"]["summary"]
    baseline_rows = [{"baseline": name, "accuracy": baseline_values[name], "n_items": 395}
                     for name in ("random", "most_frequent", "most_mined", "oracle")]
    display(HTML(result_table(baseline_rows, {"baseline": "Selection baseline", "accuracy": "Saved accuracy", "n_items": "All items"})))
    print("Baseline source:", test_comparison["baseline_path"])
    failure_rows = [r | {"response_preview": r["response"][:160]} for r in test_comparison["failures"]]
    display(HTML(result_table(failure_rows, {"system": "System", "task": "Task", "item_id": "Item",
        "status": "Status", "finish_reason": "Finish reason", "attempts": "Attempts",
        "response_preview": "Response preview", "run_id": "Original run ID"})))


### 11.4 Pilots and cumulative expenditure

Pilot records below are development observations, excluded from test scores. The additional Gemini pilot completed 20/20 logical responses in 22 calls, with two transient failed attempts recovered by the existing retry policy. All final responses reported the requested model and `STOP`; three attempts lacked complete usage, including one successful response. The original failed pilot is preserved separately.

The Gemini test was deferred on 9 October 2026 after the pilot projected approximately 579 minutes, and subsequently scheduled after the encoder, Qwen14 and xAI arms. A completed pilot does not guarantee service availability throughout a longer run. No Gemini test score is available in this comparison.

Costs use recorded token usage and the collection tariffs, plus explicit allowances when usage is incomplete. They are not settled invoices. The original 4 ILS allowance is included once. Later sessions carry earlier expenditure; session totals are therefore **not added together**. The failed initial Qwen14/xAI pilots and the separately journalled xAI HTTP diagnostic are retained, including their uncertain-cost allowances. The corrected collection version has independently inspected pilots for both added systems. The conversion allowance is 4 ILS/USD.

In [ ]:
if RUN_SAVED_TEST_ANALYSIS:
    display(HTML(result_table(test_comparison["pilots"], {
        "session": "Collection session", "system": "System", "run_id": "Pilot run ID",
        "completed": "Complete", "responses": "Planned responses", "calls": "Calls",
        "seconds": "Request time (s)", "token_cost_usd": "Token cost (USD)", "uncertain_usd": "Usage allowance (USD)",
    })))
    display(HTML(result_table(test_comparison["cost_sessions"], {
        "session": "Collection session", "prior_ils": "Carried prior (ILS)",
        "new_token_cost_ils": "New token cost (ILS)", "new_uncertain_ils": "New usage allowance (ILS)",
        "cumulative_ils": "Cumulative accounted (ILS)",
    })))
    display(HTML(result_table([test_comparison["cost"]], {
        "initial_prior_allocation_ils": "Original allowance (ILS)", "token_cost_ils": "All measured token costs (ILS)",
        "uncertain_attempt_allowances_ils": "All uncertain attempt allowances (ILS)",
        "total_accounted_ils": "Accounted total (ILS)",
    })))


### 11.5 Interpretation limits

The comparison preserves the collected systems, prompts, candidate orders, labels, scoring rules, and full denominators. The original five truncated test responses remain failures. Qwen14 additionally retains nine truncated generation responses and one HTTP failure; none is selectively replaced. The test set mixes Knesset, Wikipedia, and authored examples; automatic label agreement should not be interpreted as natural-text-only generalization or human semantic accuracy. Known label and inventory limitations are not corrected by this analysis.

Human review remains incomplete. The new trained-encoder test run uses explicitly recorded, automatically derived first-occurrence target spans; these are not human-validated annotations. The historical training-input, library and seed provenance remains incomplete despite the verified checkpoint load and new inference. Software validation of this loader establishes comparison and accounting checks; it is not additional experimental evidence. The analysis revision is recorded separately from the original collection revision.

### 11.6 Trained encoder provenance and collection status

The approved checkpoint was loaded strictly and evaluated on all 395 test items after a three-item technical validation. This is fresh inference without retraining, recorded separately from the LLM sessions. The 395 selected candidates agree individually with the historical CSV; this reproduces its choices but does not recover the original execution provenance. First-occurrence marker placement is an explicit automatic policy, not a human annotation.


In [ ]:
if RUN_SAVED_TEST_ANALYSIS:
    encoder_identity = encoder_test["manifest"]["identity"]
    display(HTML(result_table([{
        "run": encoder_test["manifest"]["run_id"],
        "revision": encoder_identity["code_revision"],
        "weights": encoder_identity["weights_sha256"],
        "target_policy": encoder_identity["target_policies"],
        "configuration": encoder_test["reconstruction"]["architecture"],
        "remaining_provenance_gaps": encoder_test["reconstruction"]["unknowns"],
    }], {"run": "Encoder test run", "revision": "Inference revision", "weights": "Checkpoint SHA-256",
         "target_policy": "Target policy", "configuration": "Architecture and encoding",
         "remaining_provenance_gaps": "Unresolved historical training provenance"})))
    status_rows = [{"system": "DictaBERT", "test": "395/395 selection; generation not applicable"}]
    for name, label in (("qwen", "Qwen2.5 7B"), ("qwen14", "Qwen2.5 14B"),
                        ("openai", "OpenAI"), ("anthropic", "Claude"),
                        ("xai", "xAI"), ("gemini", "Gemini")):
        groups = [r for r in test_comparison["metrics"] if r["system"] == name]
        pilots = [r for r in test_comparison["pilots"] if r["system"] == name]
        if groups:
            status = "; ".join(f"{r['task']}: {r['n_completed']}/{r['n_items']} technically complete" for r in groups)
        elif pilots:
            status = f"Latest pilot: {pilots[-1]['completed']}/20; no saved test run"
        else:
            status = "No collected pilot or test in this bundle"
        status_rows.append({"system": label, "test": status})
    display(HTML(result_table(status_rows, {"system": "System", "test": "Recorded collection status"})))


## 12. Human semantic review coverage

Human judgments remain separate from the automatic scores in Section 11. The review input binds each answer to its original item, task, collection run and exact response text. Historical judgments are not assigned to changed answers. Only explicitly completed reviews with a matching saved snapshot contribute judgments below; drafts remain unreviewed for this analysis.

The queue includes every automatically nonpositive answer and a separate positive-review queue. Technical failures remain identified. A human-correct answer among automatic nonpositives is evidence of a false negative for that answer; a human-wrong answer among automatic positives is evidence of a false positive. Partial and undecidable judgments are reported separately. Incomplete queue coverage cannot rule out further disagreements. No gold labels or prompts are revised from test review.

The initial 20-answer queue mixes 16 automatic nonpositives and four positives to assess usability and pace. It is not a representative evaluation sample. Any time projection below uses recorded wall-clock times, includes breaks, and is only a planning estimate.


In [ ]:
if RUN_SAVED_TEST_ANALYSIS:
    from hebrew_acronyms.test_review_export import review_coverage
    REVIEW_DIRECTORIES = [
        "human-review-test-20261009", "human-review-dictabert-test-20261009",
        "human-review-qwen14-test-20261009",
    ]
    coverage_bundles = []
    for directory in REVIEW_DIRECTORIES:
        review_root = SAVED_ARTIFACTS / directory
        if (review_root / "review-data.json").exists():
            coverage_bundles.append((directory, review_coverage(
                review_root / "review-data.json", review_root / "annotations.json")))
        else:
            print("Review input missing:", directory)
    if coverage_bundles:
        coverage_rows = [row for _, coverage in coverage_bundles for row in coverage["rows"]]
        review_runs = [(r["run_id"], r["task"]) for r in coverage_rows]
        if len(review_runs) != len(set(review_runs)):
            raise ValueError("Overlapping human-review cohorts would duplicate coverage")
        human_model_names = {m["system"]: m["model"] for m in test_comparison["models"]}
        human_model_names["dictabert"] = "Trained DictaBERT"
        human_rows = [r | {
            "model": human_model_names.get(r["system"].rsplit("_", 1)[0], r["system"]),
            "nonpositive_coverage": f"{r['automatic_nonpositive_reviewed']}/{r['automatic_nonpositive_total']}",
            "positive_coverage": f"{r['automatic_positive_reviewed']}/{r['automatic_positive_total']}",
        } for r in coverage_rows]
        display(HTML(result_table(human_rows, {
            "model": "Model", "task": "Task", "total": "Queued",
            "reviewed": "Reviewed", "not_reviewed": "Not reviewed",
            "human_undecidable": "Undecidable", "technical_failures": "Technical failures",
        })))
        display(HTML(result_table(human_rows, {
            "model": "Model", "task": "Task", "human_correct": "Human correct",
            "human_wrong": "Human wrong", "human_partial": "Partial",
            "human_undecidable": "Undecidable", "human_no_answer": "No answer",
        })))
        display(HTML(result_table(human_rows, {
            "model": "Model", "task": "Task",
            "nonpositive_coverage": "Nonpositives reviewed / all",
            "positive_coverage": "Positives reviewed / all",
            "automatic_nonpositive_human_correct": "Automatic nonpositive / human correct",
            "automatic_positive_human_wrong": "Automatic positive / human wrong",
        })))
        for directory, coverage in coverage_bundles:
            print(directory, "— judgment file included:", coverage["annotations_present"])
            print("Pace estimate:", coverage["pace"] or "Fewer than 20 completed timed reviews; no estimate.")
    else:
        print("Human-review input and judgments are not included in this bundle; coverage is not established.")
